# Quick Model Evaluation & ExplainabilityThis notebook demonstrates the Phishing Model accuracy metrics and extracts simple SHAP-like feature contributions locally.

In [ ]:
import xgboost as xgb
import joblib
import numpy as np
from features import FeatureExtractor
import math

print("Libraries loaded.")

In [ ]:
model = xgb.XGBClassifier()
model.load_model('../model/xgb_model.json')
scaler = joblib.load('../model/scaler.joblib')
extractor = FeatureExtractor()
print("Model Artifacts Loaded.")

In [ ]:
url = "http://secure-login-update-account.yolasite.com/"
raw = extractor.extract_all(url, "")
scaled = scaler.transform([raw])

p = model.predict_proba(scaled)[0][1]
print(f"Phishing Probability: {p:.4f}")

In [ ]:
# SHAP Margin extraction
booster = model.get_booster()
contribs = booster.predict(xgb.DMatrix(scaled), pred_contribs=True)[0]

feat_vals = []
for i, f in enumerate(extractor.feature_names):
    feat_vals.append((f, contribs[i], raw[i]))

feat_vals.sort(key=lambda x: abs(x[1]), reverse=True)
print(f"Base Bias Margin: {contribs[-1]:.4f}")
print("\n--- TOP CONTRIBUTING FEATURES ---")
for f, c, v in feat_vals[:5]:
    print(f"{f:20s} | SHAP: {c:+.4f} | Raw Value: {v}")

total_margin = sum([x[1] for x in feat_vals]) + contribs[-1]
sigmoid = 1 / (1 + math.exp(-total_margin))
print(f"\nSummed Sigmoid Output: {sigmoid:.4f} (Matches {p:.4f})")